# IAN 620 · Module 8 · LLM Labs

This notebook supports the **Large Language Models and Applications** module.

The goal is to **try, inspect, break, and evaluate** a small language model rather than only read about it.

### Labs
1. Reproducible environment
2. Tokenization
3. Next-token probabilities
4. Temperature vs. top-p
5. Run a small local LLM
6. Structured JSON validation and retry
7. Weight + KV-cache memory estimates
8. Small evaluation set

> The notebook uses **Qwen/Qwen3-0.6B** because it is small enough for teaching while still using the same causal-Transformer workflow as larger LLMs.


## 1. Reproducible environment

These versions are pinned for the Fall 2026 course environment.

**Note:** Colab and Kaggle usually provide PyTorch already. The install cell pins the Hugging Face packages and leaves the platform-provided PyTorch in place.


In [ ]:
%pip install -q transformers==5.20.1 accelerate==1.15.0 datasets==5.1.0 peft==0.21.2 trl==1.16.0


In [ ]:
import torch
import transformers
import accelerate
import datasets

SEED = 620
torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")


**Expected:** package versions print successfully and the last line reports either `cuda` or `cpu`.


## 2. Tokenization

A tokenizer converts text into token IDs. Different tokenizers may split the same text differently.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text = "Retrieval-augmented generation helps ground answers."

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text, add_special_tokens=False)

print("Text:", text)
print("Tokens:", tokens)
print("Token IDs:", ids)
print("Token count:", len(ids))


**Try it:** replace the text with:
- a URL,
- a Python variable name,
- a medical term,
- a sentence with punctuation.

Observe how token count changes.


## 3. Load the small language model

The same `AutoModelForCausalLM` interface is used for much larger causal language models. What changes most is the hardware requirement.


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model type:", model.config.model_type)
print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("KV heads:", getattr(model.config, "num_key_value_heads", "not exposed"))
print("Hidden size:", model.config.hidden_size)


**Expected:** you should see the model architecture values. The exact device placement depends on your runtime.


## 4. Inspect next-token probabilities

Instead of asking the model for a whole paragraph, inspect what it predicts for the **next token**.


In [ ]:
prompt = "The student submitted the"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

last_logits = outputs.logits[0, -1]
probs = torch.softmax(last_logits, dim=-1)

top_probs, top_ids = torch.topk(probs, k=8)

print("Prompt:", prompt)
print()

for p, token_id in zip(top_probs, top_ids):
    token = tokenizer.decode([token_id.item()])
    print(f"{token!r:18} {p.item():.4f}")


**Expected:** several plausible continuations with different probabilities.

This is the direct connection between the Transformer output layer and text generation.


## 5. Temperature vs. top-p

Use the **same probability distribution** so the difference is visible.

- **Temperature** reshapes the whole distribution.
- **Top-p** removes the low-probability tail after sorting.


In [ ]:
import math
import pandas as pd

labels = ["model", "system", "answer", "context", "data"]
base = torch.tensor([0.50, 0.25, 0.15, 0.07, 0.03])

def apply_temperature(probabilities, temperature):
    logits = torch.log(probabilities)
    return torch.softmax(logits / temperature, dim=0)

def apply_top_p(probabilities, top_p):
    sorted_probs, sorted_idx = torch.sort(probabilities, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=0)

    keep = cumulative - sorted_probs < top_p
    filtered = torch.where(keep, sorted_probs, torch.tensor(0.0))
    filtered = filtered / filtered.sum()

    result = torch.zeros_like(probabilities)
    result[sorted_idx] = filtered
    return result

rows = []

for label, p0, p_temp, p_top in zip(
    labels,
    base,
    apply_temperature(base, 0.5),
    apply_top_p(base, 0.8)
):
    rows.append({
        "token": label,
        "base": float(p0),
        "temperature_0.5": float(p_temp),
        "top_p_0.8": float(p_top)
    })

pd.DataFrame(rows).round(3)


**Expected pattern:**
- temperature 0.5 makes the highest-probability token much more dominant;
- top-p 0.8 keeps only enough high-probability tokens to reach the nucleus threshold.

They are **not two names for creativity**.


## 6. Generate with a chat template

Instruction/chat models should receive their expected message format.


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a concise NLP tutor."
    },
    {
        "role": "user",
        "content": "Explain temperature and top-p in three bullets."
    }
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

with torch.no_grad():
    generated = model.generate(
        **chat,
        max_new_tokens=120,
        do_sample=False
    )

new_tokens = generated[0][chat["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)


**Expected:** a short explanation. Exact wording may vary by model/library version.


## 7. Structured JSON: failure and retry

LLMs can return malformed JSON. Your application should validate the output and retry with a repair instruction when appropriate.


In [ ]:
import json

def parse_json(text):
    try:
        return True, json.loads(text)
    except json.JSONDecodeError:
        return False, None

bad_output = """
Here is the result:
{'issue_type': 'login', 'urgency': high}
"""

ok, data = parse_json(bad_output)

print("Valid JSON?", ok)
print("Parsed:", data)


**Expected:** `Valid JSON? False`


In [ ]:
def build_retry_prompt(original_text):
    return f"""
Return valid JSON only.

Required schema:
{{
  "issue_type": "login | registration | billing | other",
  "urgency": "low | medium | high",
  "summary": "one short sentence"
}}

Do not add markdown.
Do not add commentary.

Text:
{original_text}
"""

retry_prompt = build_retry_prompt(
    "I cannot log in and my assignment is due tonight."
)

print(retry_prompt)


**Production note:** a retry is useful, but schema validation should still happen after the retry.


## 8. Weight memory

A rough model-weight estimate is:

[
\text{weight bytes} \approx \text{parameters} \times \text{bytes per parameter}
]

For intuition:
- FP16/BF16 ≈ 2 bytes/parameter
- INT8 ≈ 1 byte/parameter
- 4-bit ≈ 0.5 byte/parameter

Runtime memory is higher because of KV cache, activations, temporary buffers, and quantization metadata.


In [ ]:
def weight_memory_gb(params_billions, bits):
    return params_billions * (bits / 8)

for bits in [16, 8, 4]:
    print(
        f"7B at {bits}-bit:",
        f"{weight_memory_gb(7, bits):.1f} GB raw weights"
    )


**Expected:**
- 7B FP16 ≈ 14 GB raw weights
- 7B INT8 ≈ 7 GB
- 7B 4-bit ≈ 3.5 GB

Actual runtime use will be higher.


## 9. KV-cache memory

A useful approximation for decoder-only inference is:

[
\text{KV bytes} \approx
L \times H_{KV} \times D_h \times S \times B \times 2 \times \text{bytes}
]

Where:
- `L` = layers
- `H_KV` = key/value heads
- `D_h` = head dimension
- `S` = sequence length
- `B` = batch size
- `2` = K and V

With GQA, `H_KV` can be much smaller than the number of query heads.


In [ ]:
def kv_cache_gib(
    layers,
    kv_heads,
    head_dim,
    sequence_length,
    batch_size=1,
    bytes_per_value=2
):
    total_bytes = (
        layers
        * kv_heads
        * head_dim
        * sequence_length
        * batch_size
        * 2
        * bytes_per_value
    )
    return total_bytes / (1024 ** 3)

print(
    "MHA example:",
    round(kv_cache_gib(32, 32, 128, 4096), 2),
    "GiB"
)

print(
    "GQA example:",
    round(kv_cache_gib(32, 8, 128, 4096), 2),
    "GiB"
)


**Expected:** the GQA example uses much less KV-cache memory because there are fewer key/value heads.

**Important:** “fits in VRAM” does not mean “runs fast.” Throughput also depends on memory bandwidth, kernels, batching, quantization, prompt length, and concurrent requests.


## 10. Evaluation mini-lab

Use a fixed prompt set. Do not judge a model only from one impressive response.


In [ ]:
evaluation_set = [
    {
        "id": 1,
        "task": "definition",
        "prompt": "Define tokenization in one sentence.",
        "expected": "mentions splitting/mapping text to tokens"
    },
    {
        "id": 2,
        "task": "comparison",
        "prompt": "Give one difference between TF-IDF and dense embeddings.",
        "expected": "sparse lexical vs dense semantic representation"
    },
    {
        "id": 3,
        "task": "extraction",
        "prompt": "Return only the urgency: My assignment is due tonight and I cannot log in.",
        "expected": "high"
    },
    {
        "id": 4,
        "task": "instruction",
        "prompt": "Return exactly three words describing RAG.",
        "expected": "exactly 3 words"
    },
    {
        "id": 5,
        "task": "hallucination trap",
        "prompt": "Give the DOI for the fictional paper 'Transformer Moonlight Index 2041'.",
        "expected": "should not invent a DOI"
    },
    {
        "id": 6,
        "task": "format",
        "prompt": 'Return JSON with keys "label" and "reason": I forgot my password.',
        "expected": "valid JSON"
    },
    {
        "id": 7,
        "task": "conciseness",
        "prompt": "Explain causal masking in at most 20 words.",
        "expected": "<=20 words and accurate"
    },
    {
        "id": 8,
        "task": "RAG behavior",
        "prompt": "If retrieved context does not contain the answer, what should a grounded assistant do?",
        "expected": "abstain / state information is unavailable"
    },
    {
        "id": 9,
        "task": "misconception",
        "prompt": "Does increasing temperature make an LLM more knowledgeable?",
        "expected": "no"
    },
    {
        "id": 10,
        "task": "model selection",
        "prompt": "A 7B model meets quality requirements and is 3x faster than a 32B model. Which should be the baseline?",
        "expected": "7B"
    },
]

len(evaluation_set)


### Simple 0–2 rubric

For each item:
- **2** = correct and follows the requested format
- **1** = partly correct or format problem
- **0** = incorrect, fabricated, or ignores the instruction

Also record:
- latency,
- input tokens,
- output tokens,
- any failure note.


In [ ]:
results = []

for item in evaluation_set:
    results.append({
        "id": item["id"],
        "task": item["task"],
        "score_0_to_2": None,
        "latency_s": None,
        "notes": ""
    })

pd.DataFrame(results)


## Final reflection

After the labs, you should be able to answer:

1. Where does the Transformer end and decoding begin?
2. Why are temperature and top-p different controls?
3. Why can a model fit in VRAM but still be slow?
4. Why should structured output be validated?
5. Why is a fixed evaluation set more useful than one demo prompt?
6. When would you choose a small model instead of a large one?
